# Communication Dropout Diagnostics

Run `scripts/diagnose_joint_happo.py` for a communication-dropout sweep, or provide an existing `FILELIST` of diagnostic JSON files to skip rerunning diagnostics. The notebook compares the core joint performance metrics:

- scout recall: mean and std
- confirmation recall: mean and std
- full-confirm success: rate and binary std
- final confidence: mean and std
- final coverage: mean and std

Edit the sweep settings in the first code cell. If `FILELIST` is empty, the notebook writes one JSON and one diagnostics plot per dropout value, then reads those JSONs. If `FILELIST` is non-empty, it reads those files directly and does not launch diagnostics.

In [ ]:
# Sweep configuration. Set CKPT explicitly, or leave it as None to use the newest run under MODLABEL.
MODLABEL = 'uav4_ugv3_1km_256'
CKPT = "/Users/aschuetz/Software/capstone/omnisearch/results/harl_runs/wildfire/wildfire_search/happo/uav4_ugv3_1km_256_190_900_hid128/seed-00001-2026-07-18-00-05-21/models"
NSTEPS_PER_EPISODE = 900
SEED_START = 1000
SEED_END = 1009
DROPOUTS = [0.0, 0.3, 0.5, 0.7, 0.9, 1.0]

RUN_DIAGNOSTICS = True
FORCE_RERUN_DIAGNOSTICS = True
COMMS_DROPOUT_MODE = 'bursty'
COMMS_MAP_MODE = 'per_agent'
COMMS_DROPOUT_MIN_STEPS = 5
COMMS_DROPOUT_MAX_STEPS = 15

# Optional: provide existing diagnose_joint_happo.py JSON outputs here to skip rerunning diagnostics.
#FILELIST = []
FILELIST = [
     'outputs/uav4_ugv3_1km_256_190_900_hid128__coms00_1000_1009.json',
     'outputs/uav4_ugv3_1km_256_190_900_hid128__coms03_1000_1009.json',
     'outputs/uav4_ugv3_1km_256_190_900_hid128__coms07_1000_1009.json',
     'outputs/uav4_ugv3_1km_256_190_900_hid128__coms09_1000_1009.json',
     'outputs/uav4_ugv3_1km_256_190_900_hid128__coms10_1000_1009.json',
]


In [ ]:
import json
import shlex
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CWD = Path.cwd().resolve()
if (CWD / 'scripts' / 'diagnose_joint_happo.py').is_file():
    PROJECT_ROOT = CWD
elif (CWD.parent / 'scripts' / 'diagnose_joint_happo.py').is_file():
    PROJECT_ROOT = CWD.parent
else:
    PROJECT_ROOT = Path('..').resolve()

OUTPUT_DIR = PROJECT_ROOT / 'outputs'
RESULTS_ROOT = PROJECT_ROOT / 'results/harl_runs/wildfire/wildfire_search/happo'

def _resolve_checkpoint(ckpt):
    if ckpt is not None:
        ckpt = Path(ckpt).expanduser()
        if not ckpt.is_absolute():
            ckpt = PROJECT_ROOT / ckpt
        if not ckpt.is_dir():
            raise FileNotFoundError(f'Checkpoint directory not found: {ckpt}')
        return ckpt.resolve()

    candidates = sorted((RESULTS_ROOT / MODLABEL).glob('seed-*/models'), key=lambda p: p.stat().st_mtime)
    if not candidates:
        raise FileNotFoundError(
            f'No checkpoint found under {RESULTS_ROOT / MODLABEL}. '
            'Set CKPT to the checkpoint models directory.'
        )
    return candidates[-1].resolve()


def _dropout_cli_value(dropout):
    return f'{float(dropout):g}'


def _dropout_tag(dropout):
    digits = f'{float(dropout):.2f}'.split('.')[1].rstrip('0') or '0'
    return digits.zfill(2)


def _diagnostic_output_paths(dropout):
    tag = _dropout_tag(dropout)
    stem = f'{MODLABEL}_comms{tag}_{SEED_START}_{SEED_END}'
    return OUTPUT_DIR / f'{stem}.json', OUTPUT_DIR / f'{stem}.png'



def _resolve_json_file(path):
    path = Path(path).expanduser()
    candidates = [path]
    if not path.is_absolute():
        candidates.extend([PROJECT_ROOT / path, PROJECT_ROOT / 'notebooks' / path])
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()
    return path.resolve() if path.is_absolute() else (PROJECT_ROOT / path).resolve()


CKPT = _resolve_checkpoint(CKPT)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)
print('Checkpoint:', CKPT)


In [ ]:
if FILELIST:
    JSON_FILES = [_resolve_json_file(path) for path in FILELIST]
    print('Using provided file list; diagnostics will not be run.')
else:
    JSON_FILES = []
    seeds = [str(seed) for seed in range(SEED_START, SEED_END + 1)]

    for dropout in DROPOUTS:
        json_output, plots_output = _diagnostic_output_paths(dropout)
        if RUN_DIAGNOSTICS and (FORCE_RERUN_DIAGNOSTICS or not json_output.is_file()):
            cmd = [
                sys.executable,
                str(PROJECT_ROOT / 'scripts/diagnose_joint_happo.py'),
                '--checkpoint-dir', str(CKPT),
                '--steps', str(NSTEPS_PER_EPISODE),
                '--seeds', *seeds,
                '--comms-dropout', _dropout_cli_value(dropout),
                '--comms-dropout-mode', COMMS_DROPOUT_MODE,
                '--comms-map-mode', COMMS_MAP_MODE,
                '--comms-dropout-min-steps', str(COMMS_DROPOUT_MIN_STEPS),
                '--comms-dropout-max-steps', str(COMMS_DROPOUT_MAX_STEPS),
                '--json-output', str(json_output),
                '--plots-output', str(plots_output),
            ]
            print('\nRunning:', shlex.join(cmd))
            log_path = json_output.with_suffix('.log')
            print('This can take a while, please be patient. More details can be found in', log_path)
            with log_path.open('w') as log:
                subprocess.run(
                    cmd,
                    cwd=PROJECT_ROOT,
                    check=True,
                    stdout=log,
                    stderr=subprocess.STDOUT,
                )
        else:
            print('Using existing diagnostics:', json_output.relative_to(PROJECT_ROOT))
        JSON_FILES.append(json_output)

missing = [path for path in JSON_FILES if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing diagnostic JSON files:\n' + '\n'.join(str(p) for p in missing))

print(f'Loaded file list: {len(JSON_FILES)} JSON file(s)')
for path in JSON_FILES:
    print(' -', path.relative_to(PROJECT_ROOT) if path.is_relative_to(PROJECT_ROOT) else path)


In [ ]:
def _summary_value(summary, *keys, default=float('nan')):
    for key in keys:
        if key in summary:
            return summary[key]
    return default


def _success_rate(summary):
    rate = _summary_value(summary, 'full_confirm_success_rate', 'success_rate')
    if pd.isna(rate):
        percent = _summary_value(summary, 'full_confirm_success_percent')
        rate = percent / 100.0 if not pd.isna(percent) else float('nan')
    return rate


def _success_std(summary, rate):
    if pd.isna(rate):
        return float('nan')
    episodes = _summary_value(summary, 'episodes')
    if pd.isna(episodes) or episodes <= 0:
        return float('nan')
    return float(np.sqrt(max(rate * (1.0 - rate), 0.0)))


def _label_from_payload(path, payload):
    scenario = payload.get('scenario', {})
    dropout = scenario.get('comms_dropout', None)
    mode = scenario.get('comms_dropout_mode', None)
    if dropout is None:
        return path.stem
    return f'{path.stem} | dropout={dropout:g}, mode={mode or "?"}'


records = []
for path in JSON_FILES:
    payload = json.loads(path.read_text())
    summary = payload.get('summary', {})
    scenario = payload.get('scenario', {})
    success_rate = _success_rate(summary)
    records.append({
        'file': str(path),
        'label': _label_from_payload(path, payload),
        'episodes': _summary_value(summary, 'episodes'),
        'comms_dropout': scenario.get('comms_dropout', float('nan')),
        'comms_dropout_mode': scenario.get('comms_dropout_mode', 'unknown'),
        'scout_recall_mean': _summary_value(summary, 'mean_scout_recall'),
        'scout_recall_std': _summary_value(summary, 'std_scout_recall'),
        'confirm_recall_mean': _summary_value(summary, 'mean_confirm_recall'),
        'confirm_recall_std': _summary_value(summary, 'std_confirm_recall'),
        'success_mean': success_rate,
        'success_std': _success_std(summary, success_rate),
        'success_count': _summary_value(summary, 'full_confirm_success_count'),
        'final_confidence_mean': _summary_value(summary, 'mean_final_confidence'),
        'final_confidence_std': _summary_value(summary, 'std_final_confidence'),
        'final_coverage_mean': _summary_value(summary, 'mean_final_coverage_fraction'),
        'final_coverage_std': _summary_value(summary, 'std_final_coverage_fraction'),
    })

metrics = pd.DataFrame.from_records(records)
metrics


## Compact Table

The table below formats each metric as `mean +/- std` for quick comparison.

In [ ]:
def _pm(mean, std):
    if pd.isna(mean):
        return 'n/a'
    if pd.isna(std):
        return f'{mean:.3f}'
    return f'{mean:.3f} +/- {std:.3f}'


compact = pd.DataFrame({
    'label': metrics['label'],
    'episodes': metrics['episodes'].astype('Int64'),
    'scout recall': [
        _pm(mean, std)
        for mean, std in zip(metrics['scout_recall_mean'], metrics['scout_recall_std'])
    ],
    'confirm recall': [
        _pm(mean, std)
        for mean, std in zip(metrics['confirm_recall_mean'], metrics['confirm_recall_std'])
    ],
    'success': [
        _pm(mean, std)
        for mean, std in zip(metrics['success_mean'], metrics['success_std'])
    ],
    'success count': metrics['success_count'].astype('Int64'),
    'final confidence': [
        _pm(mean, std)
        for mean, std in zip(metrics['final_confidence_mean'], metrics['final_confidence_std'])
    ],
    'final coverage': [
        _pm(mean, std)
        for mean, std in zip(metrics['final_coverage_mean'], metrics['final_coverage_std'])
    ],
}).set_index('label')

compact


## Plots

For communication-dropout sweeps, the notebook draws one combined line chart with shaded one-standard-deviation bands. If the inputs are not a numeric dropout sweep, it falls back to a grouped bar chart.

In [ ]:
METRIC_SPECS = [
    ('scout_recall', 'Scout recall', '#2f80ed', 'o'),
    ('confirm_recall', 'Confirm recall', '#f2994a', 's'),
    ('success', 'Full-confirm success', '#9b51e0', 'D'),
    ('final_confidence', 'Final confidence', '#27ae60', '^'),
    ('final_coverage', 'Final coverage', '#00a6a6', 'v'),
]

plot_df = metrics.copy()
plot_df['comms_dropout'] = pd.to_numeric(plot_df['comms_dropout'], errors='coerce')
has_dropout_x = plot_df['comms_dropout'].notna().all() and plot_df['comms_dropout'].nunique() > 1

plt.rcParams.update({
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titleweight': 'semibold',
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.frameon': False,
})

if has_dropout_x:
    plot_df = plot_df.sort_values(['comms_dropout', 'label'])
    x = plot_df['comms_dropout'].to_numpy(dtype=float)

    fig, ax = plt.subplots(figsize=(9.5, 5.6))
    for prefix, label, color, marker in METRIC_SPECS:
        mean = plot_df[f'{prefix}_mean'].to_numpy(dtype=float)
        std = plot_df[f'{prefix}_std'].to_numpy(dtype=float)
        ax.plot(
            x,
            mean,
            label=label,
            color=color,
            marker=marker,
            markersize=6.5,
            linewidth=2.4,
        )
        lower = np.clip(mean - std, 0.0, 1.0)
        upper = np.clip(mean + std, 0.0, 1.0)
        ax.fill_between(x, lower, upper, color=color, alpha=0.10, linewidth=0)

    ax.set_title('Joint Performance Under Communication Dropout', fontsize=15, pad=12)
    ax.set_xlabel('communication dropout probability')
    ax.set_ylabel('fraction')
    ax.set_ylim(0.0, 1.03)
    ax.set_xlim(max(float(np.nanmin(x)) - 0.03, -0.01), min(float(np.nanmax(x)) + 0.03, 1.01))
    ax.set_xticks(x)
    ax.grid(axis='y', color='#d0d7de', alpha=0.55, linewidth=0.8)
    ax.grid(axis='x', color='#d0d7de', alpha=0.18, linewidth=0.8)
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), title='metric')
    fig.tight_layout()
else:
    plot_df = plot_df.reset_index(drop=True)
    x = np.arange(len(plot_df))
    width = 0.15

    fig, ax = plt.subplots(figsize=(max(10, len(plot_df) * 1.4), 5.8))
    for offset, (prefix, label, color, _) in enumerate(METRIC_SPECS):
        mean = plot_df[f'{prefix}_mean'].to_numpy(dtype=float)
        std = plot_df[f'{prefix}_std'].to_numpy(dtype=float)
        positions = x + (offset - (len(METRIC_SPECS) - 1) / 2) * width
        ax.bar(
            positions,
            mean,
            width=width,
            yerr=std,
            capsize=3,
            color=color,
            alpha=0.82,
            label=label,
            error_kw={'elinewidth': 1.0, 'alpha': 0.65},
        )

    labels = plot_df['label'].str.replace(r' \| .*$', '', regex=True).tolist()
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=30, ha='right')
    ax.set_title('Joint Diagnostic Performance Comparison', fontsize=15, pad=12)
    ax.set_ylabel('fraction')
    ax.set_ylim(0.0, 1.03)
    ax.grid(axis='y', color='#d0d7de', alpha=0.55, linewidth=0.8)
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), title='metric')
    fig.tight_layout()

plt.show()
